In [7]:
from pathlib import Path

CURRENT_DIR = Path.cwd()

# Possible locations depending on where VS Code starts the notebook
candidates = [
    CURRENT_DIR / "Data" / "UCI HAR Dataset",
    CURRENT_DIR.parent / "Data" / "UCI HAR Dataset"
]

DATA_DIR = None

for path in candidates:
    if path.exists():
        DATA_DIR = path
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate the UCI HAR Dataset folder."
    )

print("Current directory:", CURRENT_DIR)
print("Dataset directory:", DATA_DIR)
print("Dataset exists:", DATA_DIR.exists())

Current directory: f:\research\Deep-Learning-Assignment\Preprocessing
Dataset directory: f:\research\Deep-Learning-Assignment\Data\UCI HAR Dataset
Dataset exists: True


In [8]:
SIGNALS = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


def load_signals(base_path, split):
    signal_data = []

    for signal in SIGNALS:
        file_path = (
            base_path
            / split
            / "Inertial Signals"
            / f"{signal}_{split}.txt"
        )

        signal_data.append(
            np.loadtxt(file_path)
        )

    return np.transpose(
        np.array(signal_data),
        (1, 2, 0)
    )

In [9]:
X_train_full = load_signals(
    DATA_DIR,
    "train"
)

X_test = load_signals(
    DATA_DIR,
    "test"
)

print(X_train_full.shape)
print(X_test.shape)

(7352, 128, 9)
(2947, 128, 9)


In [13]:
y_train_full = np.loadtxt(
    DATA_DIR / "train" / "y_train.txt",
    dtype=int
)

y_test = np.loadtxt(
    DATA_DIR / "test" / "y_test.txt",
    dtype=int
)

In [11]:
y_train_full = y_train_full - 1
y_test = y_test - 1

print(np.unique(y_train_full))
print(np.unique(y_test))

[0 1 2 3 4 5]
[0 1 2 3 4 5]


In [12]:
subjects_train_full = np.loadtxt(
    DATA_DIR / "train" / "subject_train.txt",
    dtype=int
)

subjects_test = np.loadtxt(
    DATA_DIR / "test" / "subject_test.txt",
    dtype=int
)

In [14]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    splitter.split(
        X_train_full,
        y_train_full,
        groups=subjects_train_full
    )
)

In [15]:
X_train = X_train_full[train_idx]
X_val = X_train_full[val_idx]

y_train = y_train_full[train_idx]
y_val = y_train_full[val_idx]

subjects_train = subjects_train_full[train_idx]
subjects_val = subjects_train_full[val_idx]

In [16]:
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (5551, 128, 9)
Validation: (1801, 128, 9)
Testing: (2947, 128, 9)


In [17]:
train_subjects = set(
    np.unique(subjects_train)
)

val_subjects = set(
    np.unique(subjects_val)
)

test_subjects = set(
    np.unique(subjects_test)
)

print(
    "Train ∩ Validation:",
    train_subjects & val_subjects
)

print(
    "Train ∩ Test:",
    train_subjects & test_subjects
)

print(
    "Validation ∩ Test:",
    val_subjects & test_subjects
)

Train ∩ Validation: set()
Train ∩ Test: set()
Validation ∩ Test: set()


In [18]:
mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

std = np.where(
    std == 0,
    1,
    std
)

In [19]:
X_train_normalized = (
    X_train - mean
) / std

X_val_normalized = (
    X_val - mean
) / std

X_test_normalized = (
    X_test - mean
) / std

In [20]:
print(
    "Training mean:",
    X_train_normalized.mean(
        axis=(0, 1)
    )
)

print(
    "Training std:",
    X_train_normalized.std(
        axis=(0, 1)
    )
)

Training mean: [-2.11354383e-17 -1.52078154e-17  1.87603891e-17  1.63690895e-17
 -1.80003733e-19  3.37431998e-17 -1.50494121e-15  1.24102574e-16
  5.15530691e-16]
Training std: [1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [21]:
ACTIVITY_NAMES = np.array([
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
])

output_file = (
    OUTPUT_DIR
    / "har_processed.npz"
)

np.savez_compressed(
    output_file,

    X_train=X_train_normalized,
    X_val=X_val_normalized,
    X_test=X_test_normalized,

    y_train=y_train,
    y_val=y_val,
    y_test=y_test,

    subjects_train=subjects_train,
    subjects_val=subjects_val,
    subjects_test=subjects_test,

    mean=mean,
    std=std,

    activity_names=ACTIVITY_NAMES
)

print(
    "Saved processed dataset to:",
    output_file
)

Saved processed dataset to: f:\research\Deep-Learning-Assignment\Processed_Data\har_processed.npz
